# Kimedics → Salesforce match summary

In [1]:
import os, sys, re
from pathlib import Path
from collections import defaultdict

project_root = Path.cwd().resolve()
for _ in range(15):
    if (project_root / "src" / "utils").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not locate repo root containing src/utils")

sys.path.insert(0, str(project_root / "src"))
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

# --- Load Kimedics (Supabase) ---
from utils.supabase_db import get_conn, get_job_current
with get_conn() as conn:
    kim_rows = get_job_current(conn, limit=None, schema="public")
print(f"{len(kim_rows)} Kimedics rows")

# --- Load Salesforce ---
from utils.salesforce import pull_all_jobs
TOKEN_URL = os.environ.get("SALESFORCE_TOKEN_URL") or "https://proxi.my.salesforce.com"
USE_SANDBOX = os.environ.get("SALESFORCE_USE_SANDBOX", "").lower() in ("1", "true", "yes")
USE_CC = os.environ.get("SALESFORCE_USE_USERNAME_PASSWORD", "").lower() not in ("1", "true", "yes")
sf_jobs = pull_all_jobs(
    consumer_key=os.environ["SALESFORCE_CONSUMER_KEY"],
    consumer_secret=os.environ["SALESFORCE_CONSUMER_SECRET"],
    username=os.environ.get("SALESFORCE_USERNAME") or None,
    password=os.environ.get("SALESFORCE_PASSWORD") or None,
    use_client_credentials=USE_CC,
    token_url=TOKEN_URL,
    security_token=os.environ.get("SALESFORCE_SECURITY_TOKEN") or None,
    use_sandbox=USE_SANDBOX,
)
print(f"{len(sf_jobs)} Salesforce Job__c records")

111 Kimedics rows
4457 Salesforce Job__c records


In [2]:
def norm(val):
    """Normalise practice/facility for matching.
    '4013 - Suffolk, VA (Harborview)' → '4013 suffolk va'
    '4013- Suffolk, VA'               → '4013 suffolk va'
    '4035 - Los Lunas, NM - closed'   → '4035 los lunas nm'
    """
    s = (val or "").strip().lower()
    s = re.sub(r"\(.*?\)", "", s)                # strip parenthetical like (Harborview)
    s = re.sub(r"\s*-\s*(closed|closing)\s*$", "", s)  # strip trailing - Closed
    s = re.sub(r"[,.\-–]", " ", s)               # replace punctuation with spaces (keeps store number as token)
    return re.sub(r"\s+", " ", s).strip()

def norm_date(val):
    s = (val or "").strip()
    if not s:
        return ""
    if len(s) >= 10 and s[4] == "-":
        return s[:10]
    m = re.match(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)
    if m:
        return f"{m.group(3)}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"
    m = re.match(r"(\d{1,2})/(\d{1,2})/(\d{2})$", s)
    if m:
        yr = 2000 + int(m.group(3))
        return f"{yr}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"
    return s[:10]

_ADDR_STRIP = re.compile(r"\b(st|street|ave|avenue|blvd|boulevard|dr|drive|rd|road|ln|lane|ct|court|cir|circle|way|pl|place|ste|suite|apt|unit|#)\b", re.I)
_ADDR_NOISE = re.compile(r"[.,#\-/]")

def norm_addr(val):
    """Normalise address for fuzzy comparison: lowercase, strip common suffixes/noise, collapse whitespace."""
    s = (val or "").strip().lower()
    s = _ADDR_NOISE.sub(" ", s)
    s = _ADDR_STRIP.sub("", s)
    return re.sub(r"\s+", " ", s).strip()

def addr_token_overlap(a, b):
    """Jaccard similarity on address tokens (0..1)."""
    ta = set(a.split())
    tb = set(b.split())
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

ADDR_THRESHOLD = 0.5

# ---- Build SF lookup indices ----
sf_by_practice = defaultdict(set)
sf_by_date = defaultdict(set)
sf_by_both = defaultdict(set)
sf_addr_index = []  # list of (norm_addr, sf_id) for fuzzy matching

for r in sf_jobs:
    p = norm(r.get("Job_Client_Job_Id__c"))
    d = norm_date(r.get("Job_Open_Date__c"))
    a = norm_addr(r.get("Job_Worksite_1_Address__c"))
    sid = r["Id"]
    if p:
        sf_by_practice[p].add(sid)
    if d:
        sf_by_date[d].add(sid)
    if p and d:
        sf_by_both[(p, d)].add(sid)
    if a:
        sf_addr_index.append((a, sid))

def find_addr_matches(kim_addr):
    """Return set of SF Ids whose address overlaps above threshold."""
    if not kim_addr:
        return set()
    return {sid for sa, sid in sf_addr_index if addr_token_overlap(kim_addr, sa) >= ADDR_THRESHOLD}

# ---- Strategies ----
strategy_defs = [
    ("practice only",          True,  False, False),
    ("date only",              False, True,  False),
    ("address only",           False, False, True),
    ("practice + date",        True,  True,  False),
    ("practice + address",     True,  False, True),
    ("date + address",         False, True,  True),
    ("practice + date + addr", True,  True,  True),
]

rows = []
for label, use_p, use_d, use_a in strategy_defs:
    c1, cN, c0 = 0, 0, 0
    for kr in kim_rows:
        p = norm(kr.get("practice_value"))
        d = norm_date(kr.get("posted_date"))
        a = norm_addr(kr.get("address_line"))

        needed = []
        if use_p: needed.append(p)
        if use_d: needed.append(d)
        if use_a: needed.append(a)
        if not all(needed):
            c0 += 1
            continue

        hit_sets = []
        if use_p: hit_sets.append(sf_by_practice.get(p, set()))
        if use_d: hit_sets.append(sf_by_date.get(d, set()))
        if use_a: hit_sets.append(find_addr_matches(a))

        hits = hit_sets[0]
        for hs in hit_sets[1:]:
            hits = hits & hs

        if len(hits) == 1:   c1 += 1
        elif len(hits) > 1:  cN += 1
        else:                c0 += 1

    rows.append({"strategy": label, "1:1": c1, "1:N": cN, "unmatched": c0, "total": len(kim_rows)})

print(f"{'Strategy':<28} {'1:1':>6} {'1:N':>6} {'None':>6} {'Total':>6}")
print("-" * 56)
for r in rows:
    print(f"{r['strategy']:<28} {r['1:1']:>6} {r['1:N']:>6} {r['unmatched']:>6} {r['total']:>6}")

Strategy                        1:1    1:N   None  Total
--------------------------------------------------------
practice only                   104      0      7    111
date only                        26     49     36    111
address only                     85     10     16    111
practice + date                   3      0    108    111
practice + address               88      0     23    111
date + address                    2      0    109    111
practice + date + addr            2      0    109    111


In [3]:
# Sample non-null values being compared (raw → normalised)
print(f"{'Source':<12} {'Field':<30} {'Raw → Normalised'}")
print("-" * 100)

kim_practices = [(r.get("practice_value") or "") for r in kim_rows if (r.get("practice_value") or "").strip()][:8]
kim_dates = [(r.get("posted_date") or "") for r in kim_rows if (r.get("posted_date") or "").strip()][:5]
kim_addrs = [(r.get("address_line") or "") for r in kim_rows if (r.get("address_line") or "").strip()][:5]
sf_practices = [(r.get("Job_Client_Job_Id__c") or "") for r in sf_jobs if (r.get("Job_Client_Job_Id__c") or "").strip()][:8]
sf_dates = [(r.get("Job_Open_Date__c") or "") for r in sf_jobs if (r.get("Job_Open_Date__c") or "").strip()][:5]
sf_addrs = [(r.get("Job_Worksite_1_Address__c") or "") for r in sf_jobs if (r.get("Job_Worksite_1_Address__c") or "").strip()][:5]

for label, field, vals, fn in [
    ("Kimedics", "practice_value", kim_practices, norm),
    ("Salesforce", "Job_Client_Job_Id__c", sf_practices, norm),
    ("Kimedics", "posted_date", kim_dates, norm_date),
    ("Salesforce", "Job_Open_Date__c", sf_dates, norm_date),
    ("Kimedics", "address_line", kim_addrs, norm_addr),
    ("Salesforce", "Job_Worksite_1_Address__c", sf_addrs, norm_addr),
]:
    pairs = [f"{v} → {fn(v)}" for v in vals]
    print(f"{label:<12} {field:<30} {' | '.join(pairs)}")

Source       Field                          Raw → Normalised
----------------------------------------------------------------------------------------------------
Kimedics     practice_value                 4434 - Seguin, TX → 4434 seguin tx | 4292 - Tomball, TX → 4292 tomball tx | 1307 - Battle Creek, MI → 1307 battle creek mi | 2498 - Marshalltown, IA → 2498 marshalltown ia | 1314 - Springfield, IL → 1314 springfield il | 2177 - Cullman, AL → 2177 cullman al | 4179 - Wichita (N Rock Road), KS → 4179 wichita ks | 4216 - Hendersonville, NC → 4216 hendersonville nc
Salesforce   Job_Client_Job_Id__c           2022-58405 → 2022 58405 | 2022-58812 → 2022 58812 | 2022-57867 → 2022 57867 | 2022-57455 → 2022 57455 | 3163 - Santa Fe, NM → 3163 santa fe nm | 4043- Hobbs, NM → 4043 hobbs nm | 1311- Troy, OH → 1311 troy oh | 2402- Amarillo, TX → 2402 amarillo tx
Kimedics     posted_date                    03/31/26 → 2026-03-31 | 03/30/26 → 2026-03-30 | 03/12/26 → 2026-03-12 | 03/30/26 → 2026-03-30

In [4]:
# Anchor: practice_value. For unmapped rows, show potential matches from other signals.
sf_by_id = {r["Id"]: r for r in sf_jobs}

def sf_job_nums(ids):
    """Format SF Ids as Job_Number_DJC__c list."""
    nums = [(sf_by_id[s].get("Job_Number_DJC__c") or s[:10]) for s in sorted(ids) if s in sf_by_id]
    return ", ".join(nums[:5]) + ("..." if len(nums) > 5 else "")

def sf_addrs(ids):
    """Return SF addresses for matched Ids."""
    addrs = [(sf_by_id[s].get("Job_Worksite_1_Address__c") or "") for s in sorted(ids) if s in sf_by_id]
    unique = list(dict.fromkeys(a for a in addrs if a.strip()))
    return " | ".join(unique[:3]) + ("..." if len(unique) > 3 else "")

unmapped = []
for kr in kim_rows:
    p = norm(kr.get("practice_value"))
    d = norm_date(kr.get("posted_date"))
    a = norm_addr(kr.get("address_line"))

    practice_hits = sf_by_practice.get(p, set()) if p else set()
    if len(practice_hits) == 1:
        continue  # mapped 1:1 by practice — skip

    # Potential matches via other signals
    addr_hits = find_addr_matches(a) if a else set()
    date_hits = sf_by_date.get(d, set()) if d else set()
    date_addr_hits = (date_hits & addr_hits) if (d and a) else set()

    unmapped.append({
        "kr": kr,
        "reason": "missing_practice" if not p else ("no_match" if not practice_hits else f"{len(practice_hits)}_matches"),
        "addr_hits": addr_hits,
        "date_addr_hits": date_addr_hits,
    })

print(f"Practice anchor  →  {len(unmapped)} unmapped out of {len(kim_rows)}\n")
import pandas as pd

rows_for_df = []
for row in unmapped:
    kr = row["kr"]
    rows_for_df.append({
        "job_id": kr.get("job_id") or "",
        "status": kr.get("status") or "",
        "practice_value": kr.get("practice_value") or "",
        "posted_date": kr.get("posted_date") or "",
        "by_addr": sf_job_nums(row["addr_hits"]),
        "addr_1to1": "✅" if len(row["addr_hits"]) == 1 else "❌",
        "SF_address": sf_addrs(row["addr_hits"]),
        "by_date+addr": sf_job_nums(row["date_addr_hits"]),
        "address_line": kr.get("address_line") or "",
        "view_job_link": kr.get("view_job_link") or "",
    })

df = pd.DataFrame(rows_for_df)

def make_link(url):
    if url:
        return f'<a href="{url}" target="_blank">link</a>'
    return ""

df.style.format({"view_job_link": make_link}).set_properties(**{"text-align": "left"})


Practice anchor  →  7 unmapped out of 111



,job_id,status,practice_value,posted_date,by_addr,addr_1to1,SF_address,by_date+addr,address_line,view_job_link
0,19553,Closed,"4373 - Tifton, GA",03/27/26,,❌,,,"1303 US-82, Tifton, GA",link
1,19531,Closed,"4439 - Midlothian, TX",03/23/26,JN-032026-4813,✅,"110 Eric Street, Midlothian, TX 76065",,"110 Eric Street, Midlothian, TX",link
2,19437,Closed,"4247 - Houston, TX (NW Crossing)",02/26/26,JN-042023-2127,✅,"4530 Dacoma Street, Houston, TX 77092",,"4530 DACOMA ST UNIT 500, HOUSTON TX",link
3,19453,Closed,"3185 - St. Joseph, MO",03/03/26,JN-022022-1335,✅,"5101 North Belt Hwy, St. Joseph, MO",,"5101 N BELT HWY, SAINT JOSEPH MO, St. Joseph, MO",link
4,19455,Closed,"4399 - Carson City, NV",03/03/26,JN-062025-4362,✅,"3815 South Carson Street, Carson City, NV 89701",,3815 S. Carson St. Carson City NV,link
5,19488,Closed,"3185 - St. Joseph, MO",03/10/26,JN-022022-1335,✅,"5101 North Belt Hwy, St. Joseph, MO",,"5101 N BELT HWY, SAINT JOSEPH MO, St. Joseph, MO",link
6,19523,Closed,"4140 - Suffolk, VA",03/19/26,JN-092023-2369,✅,"1930 North Main Street, Suffolk, VA 23434",,"1930 N MAIN ST, Suffolk, VA",link


## Debug: show mapping details for one `job_id`

Set `JOB_ID` below and run the next cell to see:
- raw + normalized Kimedics fields
- candidate Salesforce matches for each signal (practice/date/address)
- the combined intersections used by the notebook

In [5]:
import pandas as pd

JOB_ID = "19519"  # e.g. "19531"

if not JOB_ID.strip():
    raise ValueError("Set JOB_ID to a Kimedics job_current.job_id value")

kr = next((r for r in kim_rows if str(r.get("job_id") or "").strip() == JOB_ID.strip()), None)
if not kr:
    raise ValueError(f"No Kimedics row found for job_id={JOB_ID!r}")

# Raw + normalized signals
raw_practice = kr.get("practice_value") or ""
raw_date = kr.get("posted_date") or ""
raw_addr = kr.get("address_line") or ""

p = norm(raw_practice)
d = norm_date(raw_date)
a = norm_addr(raw_addr)

practice_hits = sf_by_practice.get(p, set()) if p else set()
date_hits = sf_by_date.get(d, set()) if d else set()
addr_hits = find_addr_matches(a) if a else set()

date_addr_hits = (date_hits & addr_hits) if (d and a) else set()
practice_date_hits = (practice_hits & date_hits) if (p and d) else set()
practice_addr_hits = (practice_hits & addr_hits) if (p and a) else set()
practice_date_addr_hits = (practice_hits & date_hits & addr_hits) if (p and d and a) else set()

print("Kimedics row")
print("- job_id:", kr.get("job_id"))
print("- status:", kr.get("status"))
print("- practice_value:", raw_practice)
print("- posted_date:", raw_date)
print("- address_line:", raw_addr)
print("- view_job_link:", kr.get("view_job_link") or "")
print("\nNormalized")
print("- practice_norm:", p)
print("- posted_date_norm:", d)
print("- address_norm:", a)

print("\nCandidate counts")
print("- practice_hits:", len(practice_hits))
print("- date_hits:", len(date_hits))
print("- addr_hits:", len(addr_hits))
print("- date+addr hits:", len(date_addr_hits))
print("- practice+date hits:", len(practice_date_hits))
print("- practice+addr hits:", len(practice_addr_hits))
print("- practice+date+addr hits:", len(practice_date_addr_hits))

# Show candidate details
sf_by_id = {r["Id"]: r for r in sf_jobs}

def _sf_rows(ids):
    out = []
    for sid in sorted(ids):
        r = sf_by_id.get(sid) or {}
        out.append(
            {
                "sf_id": sid,
                "Job_Number_DJC__c": r.get("Job_Number_DJC__c") or "",
                "Job_Status__c": r.get("Job_Status__c") or "",
                "Job_Open_Date__c": r.get("Job_Open_Date__c") or "",
                "Job_Client_Job_Id__c": r.get("Job_Client_Job_Id__c") or "",
                "Job_Worksite_1_Address__c": r.get("Job_Worksite_1_Address__c") or "",
                "Job_City__c": r.get("Job_City__c") or "",
                "Job_State__c": r.get("Job_State__c") or "",
            }
        )
    return pd.DataFrame(out)

sections = [
    ("practice_hits", practice_hits),
    ("date_hits", date_hits),
    ("addr_hits", addr_hits),
    ("date+addr", date_addr_hits),
    ("practice+addr", practice_addr_hits),
    ("practice+date", practice_date_hits),
    ("practice+date+addr", practice_date_addr_hits),
]

for name, ids in sections:
    if not ids:
        continue
    dfc = _sf_rows(ids)
    print(f"\n=== {name} ({len(dfc)}) ===")
    display(dfc.head(25))
    if len(dfc) > 25:
        print(f"(showing 25 of {len(dfc)})")

Kimedics row
- job_id: 19519
- status: Closed
- practice_value: 9387 - Phoenix, AZ (MetroCenter)
- posted_date: 03/18/26
- address_line: 2827 W PEORIA AVE, PHOENIX AZ
- view_job_link: https://u45940216.ct.sendgrid.net/ls/click?upn=u001.r-2FcZPy0W-2FImvwHCHCAYtS7vjFsbxpy07ZoTVJ7yvjWbpAlQyBojEtNqauZ-2BrhGDWOw1W9ukEDBrjSJwNR1tTDJC7RSBM3dX-2BrK1U9xwxucI-3DL6zf_MBfmFsOV5Ghr2zXFwWqbFyx60nAqw1pBezYQ5oGNCcnf8c7vYJVipxldltpe3gYAPqI-2BeJo4gf1tZ8V5ZioCigEBQX9CLihbDxrp6Hi6sDLhnuquxy21RxkxR1xqalY3MZuFdSSXJn0ZVTxGJJJ6LVdUMVTLo-2FaeT0UgDZMj6pRvKXmeCxo-2BxYQ0QH8enu61BBUWMiQEiLeFqcbtM9MI0w-3D-3D

Normalized
- practice_norm: 9387 phoenix az
- posted_date_norm: 2026-03-18
- address_norm: 2827 w peoria phoenix az

Candidate counts
- practice_hits: 1
- date_hits: 4
- addr_hits: 1
- date+addr hits: 0
- practice+date hits: 0
- practice+addr hits: 1
- practice+date+addr hits: 0

=== practice_hits (1) ===


,sf_id,Job_Number_DJC__c,Job_Status__c,Job_Open_Date__c,Job_Client_Job_Id__c,Job_Worksite_1_Address__c,Job_City__c,Job_State__c
0,a015f00000S6TgQAAV,JN-082022-1764,Closed,,"9387- Phoenix, AZ (Metrocenter)","2827 West Peoria Avenue, Phoenix, AZ 85029",Phoenix,Arizona



=== date_hits (4) ===


,sf_id,Job_Number_DJC__c,Job_Status__c,Job_Open_Date__c,Job_Client_Job_Id__c,Job_Worksite_1_Address__c,Job_City__c,Job_State__c
0,a015f00000KxSEGAA3,JN-022022-1310,Open,2026-03-18,"2387- Benton Harbor, MI","1817 M 139, Benton Harbor, MI",Benton Harbor,Michigan
1,a015f00000df0bqAAA,JN-092023-2374,Open,2026-03-18,,"8290 South Holly Street, Ste A, Centennial, CO...",Centennial,Colorado
2,a01UP00000LmFDaYAN,JN-042025-3152,Open,2026-03-18,,"3802 Colby Avenue, Ste 3, Everett, WA 98201",Everett,Washington
3,a01UP00000ccuwDYAQ,JN-032026-4808,Open,2026-03-18,,"777 White Plains Road, Scarsdale, NY 10583",Scarsdale,New York



=== addr_hits (1) ===


,sf_id,Job_Number_DJC__c,Job_Status__c,Job_Open_Date__c,Job_Client_Job_Id__c,Job_Worksite_1_Address__c,Job_City__c,Job_State__c
0,a015f00000S6TgQAAV,JN-082022-1764,Closed,,"9387- Phoenix, AZ (Metrocenter)","2827 West Peoria Avenue, Phoenix, AZ 85029",Phoenix,Arizona



=== practice+addr (1) ===


,sf_id,Job_Number_DJC__c,Job_Status__c,Job_Open_Date__c,Job_Client_Job_Id__c,Job_Worksite_1_Address__c,Job_City__c,Job_State__c
0,a015f00000S6TgQAAV,JN-082022-1764,Closed,,"9387- Phoenix, AZ (Metrocenter)","2827 West Peoria Avenue, Phoenix, AZ 85029",Phoenix,Arizona


## Final output: practice-only mapping CSV

This exports a CSV that uses **only** `practice_value` matching (the same `norm()` logic already in this notebook).

- If a Kimedics row maps to exactly 1 Salesforce Job, it is filled in `sf_job_id`.
- If it maps to 0 or >1, `sf_job_id` is left blank and you can review the candidates.

In [6]:
import pandas as pd
from pathlib import Path

# Where to save
out_dir = project_root / "data" / "exports"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "kimedics_to_salesforce_practice_only_mapping.csv"

sf_by_id = {r["Id"]: r for r in sf_jobs}

def _sf_job_num(sid: str) -> str:
    r = sf_by_id.get(sid) or {}
    return (r.get("Job_Number_DJC__c") or "").strip() or sid

def _sf_worksite_id(sid: str) -> str:
    r = sf_by_id.get(sid) or {}
    return (r.get("Job_Worksite_Location_1__c") or "").strip()

def _sf_addr(sid: str) -> str:
    r = sf_by_id.get(sid) or {}
    return (r.get("Job_Worksite_1_Address__c") or "").strip()

def _sf_practice_raw(sid: str) -> str:
    r = sf_by_id.get(sid) or {}
    return (r.get("Job_Client_Job_Id__c") or "").strip()

def _sf_practice_norm(sid: str) -> str:
    return norm(_sf_practice_raw(sid))

rows = []
for kr in kim_rows:
    job_id = str(kr.get("job_id") or "").strip()
    raw_practice = (kr.get("practice_value") or "").strip()
    kim_addr = (kr.get("address_line") or "").strip()

    p = norm(raw_practice)
    hits = sorted(sf_by_practice.get(p, set())) if p else []

    sf_job_id = hits[0] if len(hits) == 1 else ""
    candidate_nums = [_sf_job_num(s) for s in hits[:10]]
    candidate_addrs = [_sf_addr(s) for s in hits[:10]]
    candidate_practices = [_sf_practice_raw(s) for s in hits[:10]]

    rows.append(
        {
            "job_id": job_id,
            "status": (kr.get("status") or "").strip(),
            "practice_value": raw_practice,
            "practice_norm": p,
            "kimedics_address_line": kim_addr,
            "sf_match_count": len(hits),
            "sf_job_id": sf_job_id,
            "sf_job_number": _sf_job_num(sf_job_id) if sf_job_id else "",
            "sf_practice_value": _sf_practice_raw(sf_job_id) if sf_job_id else "",
            "sf_practice_norm": _sf_practice_norm(sf_job_id) if sf_job_id else "",
            "sf_worksite_location_id": _sf_worksite_id(sf_job_id) if sf_job_id else "",
            "sf_worksite_1_address": _sf_addr(sf_job_id) if sf_job_id else "",
            "sf_candidate_job_numbers": ", ".join(candidate_nums),
            "sf_candidate_practice_values": " | ".join([v for v in candidate_practices if v]) + (" | ..." if len(hits) > 10 else ""),
            "sf_candidate_addresses": " | ".join([a for a in candidate_addrs if a]) + (" | ..." if len(hits) > 10 else ""),
            "view_job_link": (kr.get("view_job_link") or "").strip(),
        }
    )

mapping_df = pd.DataFrame(rows)

print("Practice-only mapping summary")
print(mapping_df["sf_match_count"].value_counts().head(10).to_string())
print("\n1:1 mapped rows:", int((mapping_df["sf_match_count"] == 1).sum()))
print("Unmapped rows:", int((mapping_df["sf_match_count"] == 0).sum()))
print("Ambiguous rows (1:N):", int((mapping_df["sf_match_count"] > 1).sum()))

mapping_df.to_csv(out_path, index=False)
print(f"\nWrote: {out_path}")

# Preview the ambiguous/unmapped first
preview = mapping_df[mapping_df["sf_match_count"].ne(1)].copy()
preview.head(25)

Practice-only mapping summary
sf_match_count
1    104
0      7

1:1 mapped rows: 104
Unmapped rows: 7
Ambiguous rows (1:N): 0

Wrote: /Users/andylee/Desktop/projects/proxi/proxi_salesforce_automation/data/exports/kimedics_to_salesforce_practice_only_mapping.csv


,job_id,status,practice_value,practice_norm,kimedics_address_line,sf_match_count,sf_job_id,sf_job_number,sf_practice_value,sf_practice_norm,sf_worksite_location_id,sf_worksite_1_address,sf_candidate_job_numbers,sf_candidate_practice_values,sf_candidate_addresses,view_job_link
15,19553,Closed,"4373 - Tifton, GA",4373 tifton ga,"1303 US-82, Tifton, GA",0,,,,,,,,,,https://u45940216.ct.sendgrid.net/ls/click?upn...
27,19531,Closed,"4439 - Midlothian, TX",4439 midlothian tx,"110 Eric Street, Midlothian, TX",0,,,,,,,,,,https://u45940216.ct.sendgrid.net/ls/click?upn...
47,19437,Closed,"4247 - Houston, TX (NW Crossing)",4247 houston tx,"4530 DACOMA ST UNIT 500, HOUSTON TX",0,,,,,,,,,,https://u45940216.ct.sendgrid.net/ls/click?upn...
54,19453,Closed,"3185 - St. Joseph, MO",3185 st joseph mo,"5101 N BELT HWY, SAINT JOSEPH MO, St. Joseph, MO",0,,,,,,,,,,https://u45940216.ct.sendgrid.net/ls/click?upn...
58,19455,Closed,"4399 - Carson City, NV",4399 carson city nv,3815 S. Carson St. Carson City NV,0,,,,,,,,,,https://u45940216.ct.sendgrid.net/ls/click?upn...
85,19488,Closed,"3185 - St. Joseph, MO",3185 st joseph mo,"5101 N BELT HWY, SAINT JOSEPH MO, St. Joseph, MO",0,,,,,,,,,,https://u45940216.ct.sendgrid.net/ls/click?upn...
108,19523,Closed,"4140 - Suffolk, VA",4140 suffolk va,"1930 N MAIN ST, Suffolk, VA",0,,,,,,,,,,https://u45940216.ct.sendgrid.net/ls/click?upn...
